# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JustAnn1234/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

* **Selected Lane:** Lane 2 — Refresh / Content Opportunity Scoring
* **Task Type:** Binary Classification & Supervised Priority Ranking
* **Formal ML Definition:**
  We frame this problem as predicting the probability $P(Y = 1 \mid X)$ that a given content item will suffer traffic decay or remain in a high-opportunity/declining state, where $X$ is a feature vector of historical search performance, engagement signals, content age, and AI traffic footprints.

  The predicted probabilities are then transformed into a continuous **Priority Refresh Score** (0–100) to produce a ranked review queue for editorial decision-support.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* **Target / Proxy Variable:** `is_declining_label` (Binary: `1` if declining/high-risk, `0` otherwise)
* **Starter Proxy Definition:**
  In our starter playground dataset, the proxy target is defined as:
  $$Y = 1 \quad \text{if} \quad \text{trend\_direction} == \text{'down'}$$
* **Production / Capstone Target Framing (Future-Looking Window):**
  To prevent data leakage in a production setting, the target will be framed as a forward-looking window:
  $$\text{Features } (X) \text{ measured over } [T_{-90}, T_0] \implies \text{Target } (Y) \text{ defined over } [T_0, T_{+30}]$$
  A page is labeled $Y = 1$ if its organic traffic or impressions decrease by $\ge 15\%$ in the forward 30-day window ($[T_0, T_{+30}]$) relative to baseline demand.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

* **Primary Ranking Metric:** **Precision@K (specifically Precision@50 and Precision@100)**
  * *Why Precision@K?* Content teams have fixed bandwidth (e.g., reviewing 50 pages per month). Precision@50 measures the percentage of the top 50 recommended pages that actually needed a refresh. If 38 out of 50 top-ranked pages are true actionable candidates, Precision@50 = $76\%$.
* **Secondary Evaluation Metrics:**
  * **ROC AUC:** Evaluates overall model discrimination across all decision thresholds.
  * **Average Precision (PR AUC):** Evaluates ranking performance under class imbalance.
* **Baseline to Beat:** A heuristic rule-based baseline (which achieves Precision@50 $\approx 0.24$ in starter benchmarks). Our ML model aims to significantly beat this baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

* **Grain:** Each row represents **one unique published content item** (`content_id`) aggregated over a historical observation window.
* **Features ($X$):** Numeric signals (`content_age_days`, `impressions_90d`, `ctr`, `avg_position`, `ai_sessions_90d`) and categorical attributes.
* **Target Column ($Y$):** `is_declining_label` ($1 = \text{Declining/Refresh Candidate}$, $0 = \text{Stable/Growing}$).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Flexible loader: Try local paths first, fallback to raw GitHub repository
data_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

paths_to_try = [
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    data_url
]

df = None
for path in paths_to_try:
    try:
        df = pd.read_csv(path)
        print(f"Successfully loaded data from: {path}")
        break
    except Exception:
        continue

if df is None:
    raise FileNotFoundError("Could not load content_refresh_anonymized.csv")

# 1. Define the Unit of Analysis (1 row = 1 unique content item)
# Create explicit target column and inspect key feature columns
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 2. Display the dataframe slice illustrating the Unit of Analysis & Target
sample_cols = [
    'content_id',
    'content_age_days',
    'impressions_90d',
    'ctr',
    'avg_position',
    'ai_sessions_90d',
    'trend_direction',
    'is_declining_label'
]

unit_of_analysis_df = df[sample_cols].head(5)

print("\n=== UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Content Item) ===")
print(f"Dataframe Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target Distribution (is_declining_label):\n{df['is_declining_label'].value_counts(normalize=True).map('{:.1%}'.format)}")
unit_of_analysis_df

Successfully loaded data from: https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv

=== UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Content Item) ===
Dataframe Shape: 30,000 rows x 45 columns
Target Distribution (is_declining_label):
is_declining_label
1    54.2%
0    45.8%
Name: proportion, dtype: object


,content_id,content_age_days,impressions_90d,ctr,avg_position,ai_sessions_90d,trend_direction,is_declining_label
0,content_304f48230142,187,3803,0.76,10.6,0,down,1
1,content_a1fb4e703a9e,445,15320,0.05,20.3,0,down,1
2,content_9aa793d4d895,141,12581,0.09,36.5,0,down,1
3,content_331d6c4de07b,463,11751,0.49,6.2,0,stable,0
4,content_d99b7a2d90ca,263,19140,0.13,44.0,0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Fixed heuristic rules (e.g., *"Flag any page older than 180 days with low CTR"*) fail in search data for three primary reasons:

1. **Non-Linear Feature Interactions:** Content decay is multi-dimensional. A 200-day-old page with high impressions and a minor position drop may be far more urgent than a 500-day-old page with negligible demand. Rules treat thresholds rigidly, whereas ML models learn non-linear interactions across age, volume, position, and engagement.
2. **Dynamic SERP Baseline Adjustment:** Click-Through Rate (CTR) naturally varies by ranking position. ML models automatically adapt expectations based on position tiers, whereas flat rule thresholds misclassify high-position pages.
3. **Capacity Constraints & Ranking:** A fixed rule flags thousands of pages indiscriminately (e.g., flagging 13,000+ pages) without ranking them by probability or expected impact. ML outputs calibrated probabilities that enable sorting the queue by risk, ensuring human reviewers spend time on the top $K$ most valuable opportunities first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Self-Check

| Self-Check Criteria | Status | Notes |
| :--- | :---: | :--- |
| Named the ML task type | **YES** | Binary Classification & Priority Ranking |
| Named the target / proxy variable | **YES** | `is_declining_label` (and future-window framing) |
| Defined the decision success metric | **YES** | Precision@K (Precision@50) & ROC AUC |
| Showed unit of analysis as a real dataframe | **YES** | Executed code output displaying 1 row = 1 content item |
| Explained why ML beats a fixed rule | **YES** | Highlighted non-linear interactions, CTR scaling, and capacity-based ranking |

#### Submission Checklist Confirmation:
- [x] Every section above is filled — markdown thinking AND code execution
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w02_ml_task_framing.ipynb`